In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import os
from sklearn.metrics import f1_score, precision_score, recall_score, cohen_kappa_score, accuracy_score
import sys                                                                                                                           
sys.path.insert(0, "../..")
from LLM_models_interface.llm_interface import LLMJudge, load_config, load_dataset

cfg = load_config("config.yaml")
judge = LLMJudge(cfg)
traces = load_dataset(cfg) 

## LLM As a Judge Experiments

In [2]:
# Round 3 filter (remove for other datasets)                                                     
ROUND = "Round 3"
traces = [t for t in traces if t.get('round') == ROUND]

In [3]:
dirname = 'saved_results'
os.makedirs(dirname, exist_ok=True) 

results = []
for record in traces:
    trace_id = record['trace_id']
    trace_text = record['trace']
    
    if len(trace_text) + len(judge.examples) > 1048570:
            trace_text = trace_text[:1048570 - len(judge.examples)]

    try:
        response = judge.judge_trace(trace_id, trace_text)
        results.append(response)
        
        # Save the current results after each evaluation
        with open(f'{dirname}/results_checkpoint.pkl', 'wb') as f:
            pickle.dump(results, f)
            
        # Optional: Save a backup copy every 10 evaluations
        if len(results) % 10 == 0:
            with open(f'{dirname}/results_backup_{len(results)}.pkl', 'wb') as f:
                pickle.dump(results, f)
                
        print(f"Completed and saved evaluation {len(results)}/{len(traces)}")
    except Exception as e:
        print(f"Error on evaluation {len(results)}: {str(e)}")
        # Save results even if there's an error
        with open(f'{dirname}/results_checkpoint.pkl', 'wb') as f:
            pickle.dump(results, f)

No credentials found, authenticating...


c:\Users\bramn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Authenticated with default credentials 
Refreshing credentials...
Completed and saved evaluation 1/5
Using cached credentials
Completed and saved evaluation 2/5
Using cached credentials
Completed and saved evaluation 3/5
Using cached credentials
Completed and saved evaluation 4/5
Using cached credentials
Completed and saved evaluation 5/5


In [4]:
rows = [{"trace_id": r.trace_id, "model": r.model_id,                                                                                
           "tokens_in": r.tokens_in, "tokens_out": r.tokens_out,                                                                       
           "latency_s": r.latency_s, "cost_usd": r.cost_usd,                                                                           
           **r.annotations} for r in results]                                                      
pd.DataFrame(rows).to_csv(f"{dirname}/{cfg.model}.csv", index=False)

In [5]:
official_modes = ['1.1','1.2','1.3','1.4','1.5',
                  '2.1','2.2','2.3','2.4','2.5','2.6',
                  '3.1','3.2','3.3']

# Process the OpenAI evaluation results
failure_mode_results = {mode: [r.annotations[mode] for r in results] for mode in official_modes}

# Print the first few entries of each failure mode to verify
for mode, values in failure_mode_results.items():
    print(f"{mode}: {values} (total yes: {sum(values)}/{len(values)}, {round(sum(values)/len(values)*100, 2)}%)")

1.1: [0, 1, 1, 0, 0] (total yes: 2/5, 40.0%)
1.2: [0, 0, 0, 0, 0] (total yes: 0/5, 0.0%)
1.3: [1, 1, 0, 0, 1] (total yes: 3/5, 60.0%)
1.4: [1, 0, 0, 0, 0] (total yes: 1/5, 20.0%)
1.5: [0, 0, 0, 0, 1] (total yes: 1/5, 20.0%)
2.1: [1, 0, 0, 0, 0] (total yes: 1/5, 20.0%)
2.2: [0, 0, 1, 0, 0] (total yes: 1/5, 20.0%)
2.3: [0, 0, 0, 0, 0] (total yes: 0/5, 0.0%)
2.4: [1, 0, 0, 0, 0] (total yes: 1/5, 20.0%)
2.5: [0, 0, 0, 0, 0] (total yes: 0/5, 0.0%)
2.6: [1, 0, 1, 0, 0] (total yes: 2/5, 40.0%)
3.1: [0, 0, 0, 0, 0] (total yes: 0/5, 0.0%)
3.2: [0, 0, 0, 1, 0] (total yes: 1/5, 20.0%)
3.3: [1, 1, 1, 0, 1] (total yes: 4/5, 80.0%)


In [6]:
ground_truth = {code: [] for code in official_modes}

for record in traces:
    for ann in record['annotations']:
        code = re.match(r'(\d+\.\d+)', ann['failure mode']).group(1)
        if code not in official_modes:  # just skip old taxonomy modes
            continue
        votes = [ann['annotator_1'], ann['annotator_2'], ann['annotator_3']]
        label = 1 if sum(votes) >= 2 else 0
        ground_truth[code].append(label)

In [8]:
y_true_flat = []
y_pred_flat = []

for mode in official_modes:
    y_true_flat.extend(ground_truth[mode])
    y_pred_flat.extend(failure_mode_results[mode])

print(f"Model:     {cfg.model}")
print(f"Accuracy:  {accuracy_score(y_true_flat, y_pred_flat):.2f}")
print(f"Precision: {precision_score(y_true_flat, y_pred_flat):.2f}")
print(f"Recall:    {recall_score(y_true_flat, y_pred_flat):.2f}")
print(f"F1:        {f1_score(y_true_flat, y_pred_flat):.2f}")
print(f"Kappa:     {cohen_kappa_score(y_true_flat, y_pred_flat):.2f}")

Model:     claude-sonnet-4-6
Accuracy:  0.73
Precision: 0.41
Recall:    0.44
F1:        0.42
Kappa:     0.25


In [9]:
rows_fm = []                                                                                                                         
for mode in official_modes:                                                                                                          
    yt = ground_truth[mode]                                                                                                          
    yp = failure_mode_results[mode]                                                                                                  
    rows_fm.append({                                                                                                                 
        "mode": mode,                                                                                                                
        "precision": precision_score(yt, yp, zero_division=0),
        "recall":    recall_score(yt, yp, zero_division=0),
        "f1":        f1_score(yt, yp, zero_division=0),
        "support":   sum(yt),
    })
pd.DataFrame(rows_fm).set_index("mode").round(2)

,precision,recall,f1,support
mode,,,,
1.1,1.00,0.67,0.80,3
1.2,0.00,0.00,0.00,0
1.3,0.00,0.00,0.00,0
1.4,0.00,0.00,0.00,0
1.5,0.00,0.00,0.00,0
2.1,0.00,0.00,0.00,0
2.2,1.00,0.20,0.33,5
2.3,0.00,0.00,0.00,3
2.4,0.00,0.00,0.00,0


In [10]:
def bootstrap_ci(y_true, y_pred, metric_fn, n=1000, ci=0.95):
    rng = np.random.default_rng(42)
    scores = []
    idx = np.arange(len(y_true))
    yt, yp = np.array(y_true), np.array(y_pred)
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        scores.append(metric_fn(yt[s], yp[s]))
    lo = np.percentile(scores, (1 - ci) / 2 * 100)
    hi = np.percentile(scores, (1 + ci) / 2 * 100)
    return lo, hi

yt = np.array(y_true_flat)
yp = np.array(y_pred_flat)

f1_lo, f1_hi = bootstrap_ci(yt, yp, lambda a, b: f1_score(a, b, zero_division=0))
kappa_lo, kappa_hi = bootstrap_ci(yt, yp, cohen_kappa_score)

print(f"F1:    {f1_score(yt, yp, zero_division=0):.3f}  95% CI [{f1_lo:.3f}, {f1_hi:.3f}]")
print(f"Kappa: {cohen_kappa_score(yt, yp):.3f}  95% CI [{kappa_lo:.3f}, {kappa_hi:.3f}]")

F1:    0.424  95% CI [0.207, 0.611]
Kappa: 0.247  95% CI [-0.010, 0.491]
